# DT8 agent: the peatland greenhouse-gas twin over MCP

DT8 wraps the three-gas drained-peatland twin (CO2, CH4, N2O) as an MCP server, `dt8_server.py`.
CO2 and N2O come from the coupled twin; CH4 is the Section 14 configuration of the standalone CH4
notebook, with the pool of vegetation killed by flooding and without the plant-functional-type
extension.

This notebook plays the orchestrator: it launches the server over stdio, discovers its tools and
calls them, the way a multi-agent orchestrator would. No model code is imported here; every number
comes back through the MCP protocol.

Requirements: `pip install -r requirements.txt` (plus Jupyter).

In [1]:
import sys
from pathlib import Path

import numpy as np
from mcp import Client, StdioServerParameters

SERVER = Path("dt8_server.py").resolve()
params = StdioServerParameters(command=sys.executable, args=[str(SERVER)])


async def ask(tool, **args):
    """Launch DT8, call one tool and return its structured result."""
    async with Client(params) as client:
        result = await client.call_tool(tool, args)
    assert not result.is_error, result.content[0].text
    return result.structured_content

## Discover the tools

In [2]:
async with Client(params) as client:
    for t in (await client.list_tools()).tools:
        print(f"{t.name:22s} {t.description[:90]}")

list_crops             List the crops (CO2, N2O) and land-cover classes (CH4) DT8 knows.
simulate_co2           Simulate one year of CO2 fluxes (GPP, Ra, Rh, Reco, NEE) for a crop on drained peat. Optio
annual_ch4             Annual net CH4 at a mean water-table depth for a land-cover class ('reed' for rewetted pal
run_ch4_drivers        Run the CH4 model on your own daily drivers (dipwell or EO water table, Sentinel-2 LAI, so
annual_n2o             Annual N2O emission for a crop on drained peat (peat-N mineralisation plus fertiliser), un
ghg_balance            Full greenhouse-gas balance (CO2 + CH4 + N2O in t CO2e/ha/yr, AR6 GWP100) for a crop under
water_table_scenario   Water-table management scenario: three-gas balance (t CO2e/ha/yr) across mean water-table 
validate_inventory     Validate the twin against the UK Peatland Inventory (CEH report): CO2 and CH4 water-table 
get_parameters         Show the current (calibrated or learned) parameters for a site.
reset_parameters       Dis

## CO2: annual budget per crop under the drained Fenland climatology

In [3]:
catalogue = await ask("list_crops")
crops = [c["code"] for c in catalogue["crops"]]
print(f"{'crop':8s} {'GPP':>6s} {'Reco':>6s} {'NEE':>6s}  (t C/ha/yr, + = source)")
for c in crops:
    r = await ask("simulate_co2", crop=c)
    print(f"{c:8s} {r['gpp_tc_ha_yr']:6.1f} {r['reco_tc_ha_yr']:6.1f} {r['nee_tc_ha_yr']:+6.1f}")

crop        GPP   Reco    NEE  (t C/ha/yr, + = source)


WW          8.5   17.5   +9.0


Beet       13.9   18.9   +5.0


Maize      11.5   18.1   +6.7


Potato      9.5   17.6   +8.1


Grass      15.5   19.4   +3.9


Lettuce     9.9   17.7   +7.8


Celery     10.5   17.8   +7.3


Leek       10.9   17.9   +7.0


Cabbage    13.3   18.6   +5.3


Linseed     4.5   16.2  +11.7


## CH4 by land-cover class and water table (Section 12 of the CH4 notebook)

At 68 cm every class is a weak sink; crop choice matters mainly once the peat is rewetted.

In [4]:
print(f"{'class':32s} {'11.5 cm':>8s} {'30 cm':>7s} {'68 cm':>7s}  (kg CH4/ha/yr)")
for v in catalogue["ch4_vegetation"]:
    vals = [(await ask("annual_ch4", mean_wtd_m=w, vegetation=v["code"]))["ch4_kg_ha_yr"]
            for w in (0.115, 0.30, 0.68)]
    print(f"{v['name'][:32]:32s} {vals[0]:8.1f} {vals[1]:7.1f} {vals[2]:7.1f}")

class                             11.5 cm   30 cm   68 cm  (kg CH4/ha/yr)


Reed / sedge (paludiculture, cal    109.3    12.5    -1.0


Upland peatland grassland            78.4     7.2    -1.3


Grassland, grazed                   117.4    13.5    -0.9


Grassland, ungrazed                 143.0    17.7    -0.7


Winter wheat / barley               135.4    16.0    -0.8


... (6 outputs left out)

Lettuce                              97.5    10.3    -1.1


Flax                                 93.0     9.4    -1.2


Linseed                              77.0     6.9    -1.3


Fallow land                          75.7     6.7    -1.3


## Section 14: methane from vegetation killed by flooding

Flood-intolerant vegetation (sown grassland, arable crops) standing in water dies and forms a labile
litter pool. The pool starts empty when the land is rewetted and fills over a few years, so CH4 rises
from year 1 to year 2. Reed, the calibration canopy, is flood tolerant and gets no extra litter.

In [5]:
for veg in ("Grass_ungrazed", "reed"):
    for years in (0, 1, 3):
        r = await ask("annual_ch4", mean_wtd_m=0.0, vegetation=veg, years_since_rewetting=years)
        print(f"{veg:15s} year {years + 1}: base {r['ch4_base_kg_ha_yr']:6.0f} + litter "
              f"{r['ch4_litter_kg_ha_yr']:6.0f} = {r['ch4_kg_ha_yr']:6.0f} kg CH4/ha/yr "
              f"(pool {r['litter_pool_g_c_m2']:.0f} g C/m2)")

Grass_ungrazed  year 1: base    230 + litter    848 =   1078 kg CH4/ha/yr (pool 199 g C/m2)


Grass_ungrazed  year 2: base    230 + litter   1419 =   1649 kg CH4/ha/yr (pool 288 g C/m2)


Grass_ungrazed  year 4: base    230 + litter   1791 =   2020 kg CH4/ha/yr (pool 346 g C/m2)


reed            year 1: base    188 + litter      0 =    188 kg CH4/ha/yr (pool 0 g C/m2)


reed            year 2: base    188 + litter      0 =    188 kg CH4/ha/yr (pool 0 g C/m2)


reed            year 4: base    188 + litter      0 =    188 kg CH4/ha/yr (pool 0 g C/m2)


## Running CH4 on your own drivers

Here two years of synthetic drivers with weather noise: air temperature only (soil temperature is derived
with the Dolschak et al. 2015 transfer function), a rewetted reed bed in year 1 and partial drainage in
year 2, on shallow peat (validity flags).

In [6]:
rng = np.random.default_rng(42)
doy = np.tile(np.arange(1, 366), 2)
tair = 10.5 + 6.7 * np.sin(2 * np.pi * (doy - 115) / 365) + rng.normal(0, 2.0, doy.size)
dd = 0.30 * np.clip(np.sin(2 * np.pi * (doy - 172) / 365), 0, None)
wtd = np.clip(np.where(np.arange(doy.size) < 365, 0.115, 0.35) + dd - 0.10, -0.15, 1.6)
lai = 0.8 + 2.7 * np.exp(-0.5 * ((doy - 200) / 70) ** 2)
r = await ask("run_ch4_drivers", tair_c=tair.tolist(), wtd_m=wtd.tolist(), lai=lai.tolist(),
              peat_depth_m=0.35, soc_pct=25.0)
print(f"{r['net_kg_ch4_m2_yr']:.3e} kg CH4/m2/yr = {r['net_kg_ch4_ha_yr']:.1f} kg CH4/ha/yr "
      f"= {r['net_t_co2e_ha_yr']:.2f} t CO2e/ha/yr")
print(f"peak {r['peak_flux_nmol_m2_s']:.1f} nmol/m2/s; soil temperature: {r['soil_temperature_source']}")
print("fraction of days flagged:", r["flag_fractions"])

7.731e-03 kg CH4/m2/yr = 77.3 kg CH4/ha/yr = 2.09 t CO2e/ha/yr
peak 127.4 nmol/m2/s; soil temperature: air temperature via Dolschak et al. (2015), open_sod_10cm
fraction of days flagged: {'flag_wasted_peat': 1.0, 'flag_wt_below_peat': 0.1958904109589041, 'flag_low_SOC': 0.0}


## Full greenhouse-gas balance by crop (t CO2e/ha/yr, AR6 GWP100)

In [7]:
print(f"{'crop':8s} {'CO2':>6s} {'N2O':>6s} {'CH4':>6s} {'net':>6s}")
for c in crops:
    b = await ask("ghg_balance", crop=c)
    print(f"{c:8s} {b['co2_t_co2e_ha_yr']:6.1f} {b['n2o_t_co2e_ha_yr']:6.1f} "
          f"{b['ch4_t_co2e_ha_yr']:6.2f} {b['net_t_co2e_ha_yr']:6.1f}")

crop        CO2    N2O    CH4    net


WW         33.1    4.3  -0.01   37.3


Beet       18.4    4.9  -0.02   23.2


Maize      24.4    6.6  -0.02   31.0


Potato     29.8    7.2  -0.03   37.0


Grass      14.2    3.4  -0.01   17.6


Lettuce    28.7    6.2  -0.03   34.8


Celery     26.8   10.2  -0.03   36.9


Leek       25.6    9.7  -0.03   35.2


Cabbage    19.5   10.2  -0.03   29.7


Linseed    42.9    3.7  -0.03   46.5


## Water-table management: the three-gas tension and the net-GHG optimum

In [8]:
s = await ask("water_table_scenario", mean_wtd_m=[0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.7, 1.0])
print("evidence role:", s["evidence_role"])
print(f"{'WTD m':>6s} {'CO2':>6s} {'CH4':>6s} {'N2O':>6s} {'net':>6s}")
for p in s["points"]:
    print(f"{p['mean_wtd_m']:6.2f} {p['co2_t_co2e_ha_yr']:6.1f} {p['ch4_t_co2e_ha_yr']:6.1f} "
          f"{p['n2o_t_co2e_ha_yr']:6.1f} {p['net_t_co2e_ha_yr']:6.1f}")
print(f"net-GHG optimum near {s['optimum_wtd_m']:.2f} m")

evidence role: response / scenario
 WTD m    CO2    CH4    N2O    net
  0.05   -2.0    4.3    0.5    2.8
  0.10   -2.3    3.6    0.9    2.1
  0.15   -1.5    1.9    1.3    1.7
  0.20   -1.1    1.0    2.0    2.0
  0.30    1.2    0.3    4.0    5.5
  0.50   12.6    0.0    6.8   19.4
  0.70   23.4   -0.0    7.3   30.7
  1.00   30.9   -0.0    7.1   37.9
net-GHG optimum near 0.15 m


## Validation against the UK Peatland Inventory

In [9]:
v = await ask("validate_inventory")
print(f"CO2 slope {v['co2_slope_model']:.3f} vs report {v['co2_slope_report']} t CO2/cm; "
      f"cropland CO2 {v['cropland_co2_model_t']:.1f} (Tier-2 {v['cropland_co2_tier2_t']})")
for k, sk in v["ch4_skill"].items():
    print(f"CH4 {k:14s} bias {sk['bias']:6.2f}  RMSE {sk['rmse']:6.2f}  r {sk['r']:.2f}  NSE {sk['nse']:.2f}")
for row in v["ch4_tier2_table"]:
    print(f"    {row['peat_class']:20s} {row['wtd_cm']:5.1f} cm  EF {row['ef_tier2']:7.2f}  model {row['model']:7.2f}")
print(f"N2O cropland {v['n2o_cropland_model_kg_n']:.1f} (Tier-2 {v['n2o_cropland_tier2_kg_n']}); "
      f"grass {v['n2o_grass_model_kg_n']:.1f} (EF {v['n2o_grass_tier2_kg_n']}); "
      f"rewetted {v['n2o_rewetted_model_kg_n']:.1f}")

CO2 slope 0.419 vs report 0.4917 t CO2/cm; cropland CO2 27.7 (Tier-2 27.06)
CH4 eq3_5_70cm     bias   6.77  RMSE  14.33  r 0.98  NSE 0.88
CH4 eq3_20_70cm    bias   1.57  RMSE   3.56  r 1.00  NSE 0.83
CH4 tier2_classes  bias  -6.59  RMSE  22.00  r 0.98  NSE 0.83
    Near-natural fen       2.6 cm  EF  143.25  model  171.74
    Rewetted fen          11.5 cm  EF  111.44  model  109.28
    Extensive grassland   37.0 cm  EF   35.91  model    5.78
    Intensive grassland   43.0 cm  EF   29.03  model    2.83
    Cropland              68.0 cm  EF    1.96  model   -0.98
N2O cropland 16.3 (Tier-2 16.28); grass 7.9 (EF 7.39); rewetted 2.0


## Self-learning

Start CH4 and N2O from deliberately wrong priors and recover the true parameters from a synthetic
observation stream. With `persist=True` the learned set would become the site's parameters (and,
with `DT8_STATE_FILE` set, survive a server restart).

In [10]:
for gas in ("ch4", "n2o"):
    r = await ask("self_learn_synthetic", gas=gas, epochs=20)
    for k in r["learned"]:
        print(f"{gas}: {k:8s} prior {r['prior'][k]:8.3f} -> learned {r['learned'][k]:8.3f} "
              f"(true {r['truth'][k]:.3f})")

ch4: k_prod   prior  448.032 -> learned  198.152 (true 203.651)
ch4: Q10_CH4  prior    2.000 -> learned    3.537 (true 3.600)


n2o: ef_peat  prior    0.033 -> learned    0.016 (true 0.016)
n2o: Q10_n2o  prior    1.800 -> learned    2.477 (true 2.500)
